In [ ]:
import pandas as pd
import os
from config import CENSUS_INPUT_DIR, CENSUS_OUTPUT_DIR

#DATA_ROOT = Path(r"C:\Users\johnh\Desktop\Datasets\_6_Demographics\US_Census")
folder_names = os.listdir(CENSUS_INPUT_DIR)
# OR USE THIS 
# folder_names = [p.name for p in INPUT_DIR.iterdir() if p.is_dir()]
print(*folder_names, sep='\n')


replacements = {
    ":": "",
    "--":"",
    #"_years":"y",
    #"_year":"y",
    #"Mediany":"median_year",
    "2_(dollars)": "2units",
    "_(dollars)":"",
    "(in minutes)" : "",
    "Estimate!!": "",
    "Median_age!!": "",
    "Total!!": "",
    "transit!!":"",
    "United_States": "US",
    "Average_household_size!!":"",
    "Aggregate_value": "sum-value",

    "_vehicle_available":"-car-household",
    "_vehicles_available":"-car-household",
    "_or_more_vehicles_available": "+-car-household",

    "Median_year_structure_built!!":"med.-age-built_",

    "Moved_from": "",
    "different": "diff.",

    "Owner_occupied":"owners",
    "Renter_occupied":"renters",
    
    "1,_detached": "SFR",
    "1,_attached": "townhouse",
    "1,_detached_or_attached": "1unit",
    "2_to_4" : "2-4units",
    "3_or_4": "3-4units",
    "5_or_more": "5+units",
    "5_to_19" : "5-19units",
    "20_to_49" : "20-49units",
    "50_or_more" : "50+units",
    "Mobile_home" : "mobile-home",

    "travel_time_to_work" : "commute-time",
    "Worked_at_home": "WFH",
    "Other_means":"other",
    "(carro_público_in_Puerto_Rico)":"",
    "Car,_truck,_or_van" : "car",
    "Subway_or_elevated": "subway",
    "Subway_or_elevated_rail": "subway",
    "Public_transportation_(excluding_taxicab)" : "transit",
    "Long-distance_train_or_commuter_rail_or_Ferryboat" : "long-distance-rail",
    "Streetcar_or_trolley_car_(carro_publico_in_Puerto_Rico),_subway_or_elevated" : "urban-rail",
    "Bus_or_trolley_bus_2011":"bus",
    "Ferryboat":"ferry",
    

    #   "" : "",
}
#TODO: create separate replacement lists for different census tables

colDropStrings = [ # drop columns if they contain any of these strings
    "Boat,_RV,_van,_etc",
    "In_2_person_carpool",
    "In_3-or-more-person_carpool",
    "Margin_of_Error"
]

B08136_Commute_Time_By_Mode
B08203_Vehicles_Per_Worker
B08301_Commute_count_by_mode
B11001_Household_Type
B25001_Housing_Units
B25002_Occupancy_Status
B25004_Vacancy_Status
B25005_Vacant_Current_Residence_Elsewhere
B25008_Total_Population_By_Tenure
B25010_Average_Household_Size_By_Tenure
B25014_Occupants_Per_Room
B25017_Room_Count
B25018_Median_Room_Count
B25019_Aggregate_Room_Count
B25021_Median_Room_Count_By_Tenure
B25024_Units_In_Structure
B25031_Median_Rent_By_Bedroom_Count
B25032_Tenure_By_Units_In_Structure
B25037_Median_Building_Age_By_Tenure
B25039_Median_Duration_Lived_By_Tenure
B25044_Vehicles_Available_By_Tenure
B25052_Kitchen_Facilities
B25058_Median_Contract_Rent
B25064_Median_Gross_Rent
B25066_Aggregate_Gross_Rent_By_Units_In_Structure
B25069_Inclusion_of_Utilities_In_Rent
B25071_Median_Gross_Rent_Burden
B25075_Value
B25077_Median_Value
B25080_Aggregate_Value_By_Units_In_Structure
B25105_Median_Monthly_Housing_Costs
B25111_Median_Gross_Rent_By_Decade_Built
B25113_Median_G

In [ ]:
#John Nabors Feb 20, 2026
singleFolderOverride = False #allows manual processing of individual datasets
singleFolderName = "B25080_Aggregate_Value_By_Units_In_Structure"

for folder_name in folder_names: #loop all folders in /input
    if singleFolderOverride: folder_name = singleFolderName
    print("parsing folder: ", folder_name)

    folder_path = CENSUS_INPUT_DIR / folder_name
    #data_filenames = [file for file in os.listdir(folder_path) if file.endswith("Data.csv")]
    #print(*data_filenames, sep='\n')
    data_filepaths = [p for p in folder_path.glob("*-Data.csv")]
    #print(*data_filepaths, sep='\n')
    
    
    allYears = pd.DataFrame()
    colNames = pd.DataFrame()
    for csv in data_filepaths: #loop all csvs in a folder.
        try   : df = pd.read_csv(csv, low_memory=False)
        except: print(f"Error reading csv: {csv}")
        
        year = str(csv).split('ACSDT5Y')[1][:4] #Only works with ACS 5y data.
        #print("Data Year: ", year)
        

        # clean up column name formatting.
        df = df.drop(columns=["GEO_ID"]) #remove GEOID
        df.columns = df.iloc[0] #swap column headers with the first row column descriptors/human readable names
        #print(df.columns)
        df = df[1:].reset_index(drop=True) #not sure why i need this but too lazy to figure out.
        if pd.isna(df.columns[-1]):
            df = df.drop(df.columns[-1], axis=1) #drop the na column artifact (WHY DOES IT APPEAR??)
        df.columns = (df.columns
            .str.strip()
            .str.replace(" ", "_")
        )

        # drop columns containing any of the strings in colDropStrings
        
        for drop_str in colDropStrings: # drops Margin of Error, and anything else I dont want to have in final data
            df = df.loc[:, ~df.columns.str.contains(drop_str, regex=False, na=False)]
        

        #begin column replacements
        #print(df.columns)
        for old, new in replacements.items():
            #print(old)
            oldCols = df.columns
            df.columns = df.columns.str.replace(old, new, regex=False)
            newCols = df.columns
            
        #print(df.columns)
        #df.columns = df.columns.str.replace('|'.join(replacements.keys()), lambda m: replacements[m.group()], regex=True)
        #df.columns = df.columns.str.replace('|'.join(replacements.keys()), lambda m: replacements[m.group()], regex=True)
        #df.columns = df.columns.str.replace('|'.join(replacements.keys()), lambda m: replacements[m.group()], regex=True) # in case some are missed
        df.columns = df.columns + '_' + year

        df.rename(columns={f'Geographic_Area_Name_{year}': 'Geographic_Area_Name'}, inplace=True)    
        
    

        if len(allYears) == 0:  allYears = df.copy()
        else: allYears = allYears.merge(df, on='Geographic_Area_Name', how='outer')
    

    allYears['zipCode'] = allYears['Geographic_Area_Name'].str[-5:]
    allYears = allYears[allYears['zipCode'].astype(int) > 1000] #removes puerto rico and territory zip codes
    allYears = allYears[['Geographic_Area_Name', 'zipCode'] + sorted([col for col in allYears.columns if col not in ['Geographic_Area_Name', 'zipCode'] ])]
    
    print("AllYears: ", allYears.columns)

    output_name = f"Census/Output/{folder_name}_joined.csv"
    print("outputting at: ",output_name)
    allYears.to_csv(output_name, index=False) #TODO use zip code as index


    if singleFolderOverride: break

parsing folder:  B08136_Commute_Time_By_Mode
AllYears:  Index(['Geographic_Area_Name', 'zipCode',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2011',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2012',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2013',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2014',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2015',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2016',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2017',
       'Aggregate_commute-time_(in_minutes)!!Taxicab,_motorcycle,_bicycle,_or_other_means_2018',
       ...
       'Aggregate_commute-time_(in_minutes)_2014',
       'Aggregate_commute-time_(in_minutes)_2015',
       'Aggre